## Core Assumptions

This notebook would talk about the core assumptions of the black sholes merton model and mathematically show why these assumptions exist and where do they stem from. And the caveats that come along with using the black scholes merton for real world scenarios

## Assumption 1 : The risk-free rate $r$ is known and constant

**What it says.** There is a single, known rate $r$ at which one can
borrow and lend without risk, and it does not change over the life of
the option $[0, T]$.

**Why we assume it.** For short-dated options on liquid underlyings,
$r$ changes very little over the option's life. The approximation is
excellent, and the closed-form solution is fast and accurate.

**Reality.** $r$ is set by central banks and moves with the macro
cycle. Over months, it drifts. Over years, it can move dramatically.

In [ ]:
import numpy as np
import plotly.graph_objects as go

# --- Historical Fed Funds rate (annual averages, simplified) ---
# Source: Federal Reserve Economic Data (FRED), series FEDFUNDS
years = np.array([1954, 1957, 1960, 1965, 1970, 1973, 1975, 1978, 1980,
                  1981, 1983, 1985, 1988, 1990, 1992, 1994, 1997, 2000,
                  2003, 2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020,
                  2022, 2024])

rates = np.array([0.80, 3.00, 2.50, 4.10, 7.20, 8.70, 5.80, 7.90, 13.40,
                  16.40, 9.10, 8.10, 7.60, 8.10, 3.50, 4.20, 5.50, 6.20,
                  1.10, 5.00, 1.90, 0.20, 0.15, 0.10, 0.40, 1.80, 0.40,
                  2.20, 5.30])
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=years, y=rates,
    mode="lines+markers",
    line=dict(color="#2563eb", width=2.5),
    marker=dict(size=6, color="#2563eb"),
    fill="tozeroy",
    fillcolor="rgba(37, 99, 235, 0.10)",
    name="Fed Funds rate",
    hovertemplate="Year %{x}<br>Rate %{y:.2f}%<extra></extra>",
))

fig.update_layout(
    title="US Fed Funds rate, 1954–2024 — anything but constant",
    xaxis_title="Year",
    yaxis_title="Rate (%)",
    height=500,
)

fig.show()

## Assumption 2 : Log returns are normally distributed

**What it says.** Log returns over any horizon
$[0, T]$ is normally distributed:

$$
\ln\frac{S_T}{S_0}
\;\sim\;
N\!\left(\left(\mu - \tfrac{1}{2}\sigma^2\right)T, \;\; \sigma^2 T\right).
$$

Equivalently, the stock price $S_T$ is lognormally distributed.

**Where it comes from.** This is not an independent assumption — it
is a *consequence* of the GBM dynamics. Starting from

$$
dS = \mu S \, dt + \sigma S \, dB_t,
$$

apply Itô's lemma to $f(S) = \ln S$. The second derivative
$f_{SS} = -1/S^2$ produces the $-\tfrac{1}{2}\sigma^2$ correction:

$$
d(\ln S) = \left(\mu - \tfrac{1}{2}\sigma^2\right) dt + \sigma \, dB_t.
$$

Integrating from $0$ to $T$:

$$
\ln\frac{S_T}{S_0}
=
\left(\mu - \tfrac{1}{2}\sigma^2\right)T + \sigma B_T.
$$

Since $B_T \sim N(0, T)$, the right-hand side is a linear combination
of a normal and a contant, hence normal. So normality of log returns is inherited
directly from the normality of Brownian increments.

**Reality.** Empirical log returns are not normal. They exhibit:

- **Negative skewness** : large negative moves are more common than
  large positive moves.
- **Excess kurtosis (fat tails)** : extreme events happen far more
  often than the normal distribution predicts.

For the S&P 500 daily returns, excess kurtosis is typically between
$10$ and $30$, and skewness is negative. Under a normal distribution,
a 5 sigma daily move should occur roughly once every $10{,}000$ years.
In reality, it occurs roughly once every few years.

**How to relax it.** Jump-diffusion models (Merton) add discontinuous
jumps to the price path. Lévy processes replace Brownian motion with a
heavier-tailed driver. Stochastic volatility models (Heston) make the
variance itself random, which generates fat tails endogenously.

## Assumption 3 : Volatility $\sigma$ is constant

**What it says.** The diffusion coefficient in the stock price
equation is a fixed number:

$$
dS = \mu S \, dt + \sigma S \, dB_t, \qquad \sigma = \text{constant}.
$$

This means the annualized variance rate of the stock is $\sigma^2$,
regardless of time, price level, or market regime.

**Where it comes from.** In the general form of Itô's lemma, the
second-order term is

$$
\tfrac{1}{2} f_{SS} \, (dS)^2 = \tfrac{1}{2} f_{SS} \, \sigma^2 S^2 \, dt.
$$

The step $(dS)^2 = \sigma^2 S^2\,dt$ requires that $\sigma$ is not
itself random. If $\sigma$ depended on $t$ or $S$, or on its own
Brownian driver $dW_t$, then $d\sigma$ would contribute an extra
$dW_t$ to $d\Pi$, the hedge would not be riskless, and the close form that was derived would be lost.

**Why we assume it.** Constant $\sigma$ gives a closed form price, a
single number to quote, and clean Greeks. For ATM,
short-dated options, it is a good first approximation.

**Reality.** Volatility is not constant. It exhibits:

- **Clustering.** Calm periods and wild periods persist for weeks or
  months.
- **Leverage effect.** Volatility rises when prices fall, falls when
  prices rise.
- **Smile and skew.** Implied volatility varies systematically across
  strikes and maturities.

If BSM were correct, all options on the same underlying with the same
maturity would have the same implied volatility. The fact that they do
not ,the smile is the market's direct refutation of Assumption 3.


## Assumption 4 : No dividends 
The stock pays no cash
distributions over the option's life. If it did, the stock price would
drop on the ex-dividend date, and the hedge would need to account for
the cash received. Relaxing this adds a continuous dividend yield $q$
to the PDE:

$$
f_t + \tfrac{1}{2}\sigma^2 S^2 f_{SS} + (r - q) S f_S - rf = 0.
$$

*Reality:* Most equities pay dividends. The correction is small but
non-zero, and it is standard practice to adjust for it.

## Assumption 5 : Short selling is allowed  
To form the hedge
$\Pi = -f + f_S S$, we must be able to short either the stock or the
derivative. If short selling is restricted, the portfolio cannot be
constructed, and no arbitrage only provides bounds on the price rather
than a unique value.

*Reality:* Short selling is allowed in most liquid markets, but with
constraints

## Assumption 6 : Continuous trading and no arbitrage
No arbitrage
forces $d\Pi = r\Pi\,dt$. Continuous trading lets us rebalance the
hedge at every instant, keeping $f_S$ exact. Relaxing either breaks
the perfect hedge: with discrete rebalancing, the portfolio retains
residual Gamma PnL.

*Reality:* Trading is discrete and costly. Hedging error is real, and
it is the source of the PnL that options traders actually earn or lose.